# QLoRA with PyTorch + Hugging Face
## Beginner practice project: IMDb sentiment classification

**Goal:** classify movie reviews as Negative (`0`) or Positive (`1`) using QLoRA.

QLoRA means **Quantized Low-Rank Adaptation**:
1. Load the pretrained model's supported linear weights in 4-bit.
2. Keep the quantized base model frozen.
3. Add small trainable LoRA matrices (A and B).
4. Train the adapters and task-specific classification head.

We use PyTorch underneath, Transformers to load and train the model, bitsandbytes for 4-bit quantization, and PEFT for LoRA.

This notebook uses DistilBERT for a familiar IMDb classification exercise. QLoRA is especially popular for large generative language models, but the same 4-bit-base-plus-LoRA idea is practised here.

## Cell 1 — Install libraries
Install Transformers, Datasets, PEFT, bitsandbytes, and Accelerate. bitsandbytes provides the 4-bit quantized layers.

In [ ]:
!pip -q install -U transformers datasets accelerate peft bitsandbytes

## Cell 2 — Import libraries and check the GPU

PyTorch handles tensors and the training backend. QLoRA with bitsandbytes normally needs a supported NVIDIA CUDA GPU. In Colab, choose a T4 GPU runtime.

In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from peft import (
    LoraConfig,
    TaskType,
    get_peft_model,
    prepare_model_for_kbit_training,
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Select a supported NVIDIA GPU runtime before running the 4-bit cells.")

## Cell 3 — Project settings

`MODEL_NAME` selects the pretrained model. `MAX_LENGTH` limits review length. `BATCH_SIZE` controls reviews per batch. `RANK` and `LORA_ALPHA` control the LoRA adapter.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"

MAX_LENGTH = 128
BATCH_SIZE = 8
EPOCHS = 1
LEARNING_RATE = 2e-4

RANK = 4
LORA_ALPHA = 8
LORA_DROPOUT = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## Cell 4 — Load a small IMDb dataset

IMDb reviews have labels: `0` = Negative and `1` = Positive. We use 2,000 training reviews and 500 test reviews to keep the exercise small. Test reviews are not used to update the model.

In [ ]:
dataset = load_dataset("imdb")

train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))
test_dataset = dataset["test"].shuffle(seed=42).select(range(500))

print(train_dataset[0]["text"][:300])
print("Label:", train_dataset[0]["label"])
print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

## Cell 5 — Load the tokenizer

The tokenizer converts ordinary text into `input_ids` and `attention_mask`. It uses the vocabulary and tokenization rules associated with pretrained DistilBERT; IMDb does not teach it a new vocabulary.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

## Cell 6 — Tokenize the reviews

`truncation=True` cuts off reviews longer than `MAX_LENGTH`. `batched=True` processes several reviews per mapping call. We remove the original text because the model uses token IDs, attention masks, and labels.

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_tokens = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)
test_tokens = test_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
)

print(train_tokens[0].keys())

## Cell 7 — Create the padding collator

Reviews have different lengths. `DataCollatorWithPadding` pads examples in each batch to the length of the longest review in that batch. This is dynamic padding.

In [ ]:
collator = DataCollatorWithPadding(tokenizer=tokenizer)

## Cell 8 — Configure 4-bit quantization

- `load_in_4bit=True`: load supported linear weights in 4-bit.
- `nf4`: NormalFloat 4-bit quantization, commonly used for QLoRA.
- `use_double_quant=True`: additionally quantizes quantization constants to save memory.
- `compute_dtype`: data type used for calculations. FP16 is a practical choice for many T4 setups.

The base weights are stored in a compressed 4-bit format; the model still performs calculations. This cell requires a supported CUDA GPU.

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "QLoRA with bitsandbytes needs a supported CUDA GPU. "
        "In Colab choose Runtime > Change runtime type > T4 GPU."
    )

compute_dtype = torch.float16

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print(quantization_config)

## Cell 9 — Load DistilBERT in 4-bit

We load pretrained DistilBERT with a two-class classification head. `quantization_config` applies bitsandbytes 4-bit loading, and `device_map="auto"` places the model on the available device.

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=quantization_config,
    device_map="auto",
)
print("4-bit model loaded.")

## Cell 10 — Prepare the quantized model for k-bit training

`prepare_model_for_kbit_training` performs PEFT's preparation steps for training adapters on a low-bit model. It does not add LoRA yet. The quantized base model stays frozen.

In [ ]:
base_model = prepare_model_for_kbit_training(base_model)

## Cell 11 — Define LoRA configuration

- `r=4`: rank of the small LoRA matrices.
- `lora_alpha=8`: scaling setting.
- `lora_dropout=0.1`: dropout in the LoRA branch.
- `target_modules`: add LoRA to DistilBERT Query and Value projection layers.
- `modules_to_save`: keep the task-specific classification layers trainable and save them with the adapter.

For a 768-dimensional projection and rank 4, the LoRA path is conceptually `768 → 4 → 768`.

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_lin", "v_lin"],
    modules_to_save=["pre_classifier", "classifier"],
)

## Cell 12 — Apply LoRA to the 4-bit model

`get_peft_model` attaches the LoRA adapters. The base weights remain frozen; the LoRA parameters and the task-specific classification layers are trainable. The printed summary shows the trainable parameter count.

In [ ]:
qlora_model = get_peft_model(base_model, lora_config)
qlora_model.print_trainable_parameters()

## Cell 13 — Create an accuracy function

The model returns two logits per review. `argmax` chooses the class with the larger score. We compare predictions with the correct labels and calculate accuracy.

In [ ]:
def compute_accuracy(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": float((predictions == labels).mean())}

## Cell 14 — Configure Hugging Face Trainer

`TrainingArguments` stores the training settings. Trainer manages the standard PyTorch training loop for us. `fp16=True` enables half-precision training on the supported GPU.

In [ ]:
training_args = TrainingArguments(
    output_dir="./qlora_imdb_results",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=25,
    eval_strategy="epoch",
    save_strategy="no",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=qlora_model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=test_tokens,
    data_collator=collator,
    compute_metrics=compute_accuracy,
)

## Cell 15 — Train QLoRA

This starts training. Trainer gets batches, runs the 4-bit model plus LoRA adapters, calculates loss from the IMDb labels, backpropagates, and updates the trainable adapter and classification parameters. The frozen base weights are not updated.

In [ ]:
train_result = trainer.train()
print(train_result)

## Cell 16 — Evaluate the model

`trainer.evaluate()` runs on the held-out test dataset. It reports evaluation loss and accuracy. The test set is used for checking performance, not for training updates.

In [ ]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

## Cell 17 — Predict with our own review

The function accepts normal text, tokenizes it, and asks the trained QLoRA model for a prediction. `torch.no_grad()` disables gradient calculation because we are only predicting.

In [ ]:
def predict_qlora(review):
    qlora_model.eval()
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_device = qlora_model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = qlora_model(**inputs).logits

    label = logits.argmax(dim=-1).item()
    return "Positive" if label == 1 else "Negative"


print(predict_qlora("This movie was wonderful. I really enjoyed it!"))
print(predict_qlora("The story was boring and the acting was terrible."))

## Cell 18 — Save the QLoRA adapter

`save_pretrained` saves the PEFT adapter and its configuration, not a separate full copy of the base model. We also save the tokenizer. The adapter must later be paired with the matching base model.

In [ ]:
ADAPTER_PATH = "./qlora_imdb_adapter"

qlora_model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("QLoRA adapter saved to:", ADAPTER_PATH)

## Cell 19 — Reload the saved adapter

For inference, load the same base model in 4-bit and attach the saved adapter. We do not need to call `prepare_model_for_kbit_training` when we are only loading for inference.

In [ ]:
from peft import PeftModel

fresh_base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    quantization_config=quantization_config,
    device_map="auto",
)

reloaded_qlora_model = PeftModel.from_pretrained(
    fresh_base_model,
    ADAPTER_PATH,
)
reloaded_qlora_model.eval()

print("Saved QLoRA adapter reloaded.")

## Cell 20 — Test the reloaded model

This checks that the saved adapter can be loaded and used for a new prediction.

In [ ]:
def predict_reloaded(review):
    inputs = tokenizer(
        review,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
    )
    input_device = reloaded_qlora_model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}

    with torch.no_grad():
        logits = reloaded_qlora_model(**inputs).logits

    label = logits.argmax(dim=-1).item()
    return "Positive" if label == 1 else "Negative"


print(predict_reloaded("A beautiful film with excellent performances."))

# Part 4 — Summary and practice

## Complete QLoRA data flow

`IMDb review → Tokenizer → input_ids + attention_mask → 4-bit DistilBERT + LoRA → Classification head → Positive / Negative`

## What is frozen and what is trained?

| Part | What happens |
|---|---|
| DistilBERT base weights | Loaded in 4-bit and frozen |
| LoRA A and B | Trainable |
| Classification layers | Trainable and saved with the adapter |
| Tokenizer | Converts text to token IDs; it is not trained here |

## LoRA versus QLoRA

| LoRA | QLoRA |
|---|---|
| Base model commonly loaded in higher precision | Base model loaded in 4-bit |
| Base model frozen | Quantized base model frozen |
| Train small LoRA adapters | Train small LoRA adapters |
| Less training memory than full fine-tuning | Further reduces memory needed to store base weights |

**Important:** The base model's supported linear weights are quantized to 4-bit. The trainable LoRA adapters are not necessarily 4-bit.

## Practice exercises

1. Change `RANK = 4` to `RANK = 8`. What happens to trainable parameters?
2. Change `LORA_ALPHA = 8` to `LORA_ALPHA = 16`. How does scaling change?
3. Change `target_modules=["q_lin", "v_lin"]` to `target_modules=["q_lin"]`. Which projection receives LoRA?
4. Increase training data from 2,000 to 5,000 reviews.
5. Increase epochs from 1 to 2. Compare loss and accuracy.
6. Compare this QLoRA notebook with your previous LoRA notebook.

## Troubleshooting

- Use a supported NVIDIA GPU, such as a Colab T4.
- If bitsandbytes reports a CUDA or installation error, restart the runtime after installation and run from the top.
- If your Transformers version does not recognize `eval_strategy`, update Transformers or use the evaluation argument name supported by that version.
- This is a small learning experiment, not a benchmark. Results depend on the model, subset, GPU, and settings.